In [2]:
from openai import OpenAI

client = OpenAI(api_key="USE_YOUR_OWN_KEY")

In [3]:
import os
import pandas as pd

from os       import makedirs
from tqdm     import tqdm
from datasets import load_dataset

# prepare data
dataset = load_dataset("arch-be/brabant-xvii", name="doc_by_doc")
train   = dataset['train'].to_pandas()
valid   = dataset['valid'].to_pandas()
test    = dataset['test'].to_pandas()
dataset = pd.concat([train, test, valid]).set_index(['project', 'file_id'])
# start with the longest texts (fail fast)
dataset.sort_values(by='text', key=lambda x: x.str.len(), ascending=False, inplace=True)

with open('prompt.txt', encoding='utf8') as f:
    prompt = f.read()


/home/xgillard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating valid split: 100%|██████████████████████████████████████████████████| 67/67 [00:00<00:00, 8602.78 examples/s]


In [4]:
# run the inference pipeline
model = "gpt-4o-mini"

makedirs(f'generated_texts/{model}/', exist_ok=True)
#
for id, text in tqdm(dataset['text'].items()):
    project,file_id = id
    fname = f'generated_texts/{model}/{project}_{file_id}'
    if os.path.exists(fname):
        continue
    else:
        with open(fname, 'w', encoding='utf8') as f:
            messages = [
                    {'role': 'system', 'content': prompt},
                    {'role': 'user',   'content': text},
                ]
            response = client.chat.completions.create(
                model=model,
                response_format={'type': 'json_object'},
                messages = messages,
            )
            print(response, file=f)

419it [1:42:53, 14.74s/it]
